In [0]:
from pyspark.sql import functions as F 

In [0]:
yellow_taxi_trip_silver_df= spark.read.table('yellow_taxi_trip_silver')

In [0]:
weather_df= spark.read.table('weather_silver')

In [0]:
daily_metrics= yellow_taxi_trip_silver_df.groupby('date')\
    .agg(F.sum('total_amount').alias('total_revenue'), 
         F.count('*').alias('trip_count'),
         F.avg('trip_distance').alias('avg_trip_distance'),
         F.avg('fare_amount').alias('avg_fare_amount'))
    
daily_metrics.write.mode('overwrite').format("delta").saveAsTable('daily_metrics')

In [0]:
taxi_zone_df = spark.read.table('taxi_zone_silver')

In [0]:
zone_performance_df = yellow_taxi_trip_silver_df.alias('t')\
    .join(taxi_zone_df.alias('z'), F.col('t.pickup_location_id') == F.col('z.LocationID'),"inner")\
    .groupBy(
        F.col("z.Borough"), F.col("z.Zone")
    )\
    .agg(F.sum('t.total_amount').alias('total_revenue'), 
        F.count('*').alias('trip_count'),
        F.avg('t.trip_distance').alias('avg_trip_distance'),
        F.avg('t.fare_amount').alias('avg_fare_amount'))\
        .orderBy(F.col('total_revenue').desc())

zone_performance_df.write.mode('overwrite').format("delta").saveAsTable('zone_performance')
    

In [0]:
hourly_activity_df = yellow_taxi_trip_silver_df.groupBy(F.col('hour'))\
    .agg(
    F.count('*').alias('trip_count'),
    F.avg('trip_distance').alias('avg_trip_distance')
)
    
hourly_activity_df.write.mode('overwrite').format("delta").saveAsTable('hourly_activity')

In [0]:
weather_impact_df = yellow_taxi_trip_silver_df.alias('t')\
    .join(weather_df.alias('w'), (F.col('t.date') == F.col('w.date')) & (F.col('t.hour') == F.col('w.hour')),"inner")\
    .groupBy(F.col('t.date'), F.col('t.hour'))\
    .agg(F.count('*').alias('trip_count'),
         F.avg('t.trip_distance').alias('avg_trip_distance'),
         F.avg('w.temperature_2m').alias('temperature'),
         F.avg('w.precipitation').alias('precipitation')
         )
    
weather_impact_df.write.mode('overwrite').format("delta").saveAsTable('weather_impact')